# S02 · Subgraph Matching, MCS, and RDKit vs NetworkX


## Aim of this talktorial

In **S01**, we built the foundations: **typed molecular graphs** and **typed morphisms**
(isomorphisms, automorphisms) as the matching backbone for graph transformation.

This talktorial (**S02**) moves to the *workhorse alignments* that appear throughout reaction modeling
and rule-based systems (e.g., DPO-style rewriting):

1. **Subgraph matching** — finding when a *pattern* graph occurs inside a *host* graph
   via **typed subgraph monomorphisms** (a.k.a. subgraph isomorphisms in practice).
2. **MCS alignment** — using **maximum common substructure (MCS)** as a pragmatic alignment primitive
   when an exact pattern is unknown or when molecules differ by edits.

We keep implementations minimal and transparent, using:

- **NetworkX** for explicit, attribute-based subgraph morphisms and symmetry-aware deduplication,
- **RDKit** for chemistry-aware substructure/MCS behavior (sanitization, aromaticity, valence, MCS).

**Data example:** `data/molecules.csv`

---

## Learning outcomes

After completing this talktorial, you will be able to:

- Formulate **typed subgraph matching** as an **injective typed morphism**
  \(\varphi: V(P) \hookrightarrow V(H)\) and understand what is (and is not) preserved.
- Compute **subgraph isomorphisms** (pattern → host) in **NetworkX**, and interpret multiple matches.
- Apply **symmetry-aware deduplication** of matches using automorphism or orbit ideas (why “24 matches” may mean “2 unique placements”).
- Use **RDKit substructure matching** and understand when it differs from a pure graph matcher.
- Compute **MCS with RDKit** and turn it into an **alignment map** between two molecules.
- Compare **RDKit vs NetworkX** matching behavior and attribute choices, and decide which is appropriate for
  later steps (rule extraction, reaction center localization, DPO rule application).

---

## Outline

0. **Setup & data**
1. **Typed subgraph morphisms: definition and intuition (S01 recap)**
2. **NetworkX subgraph isomorphism (pattern → host)**
3. **Symmetry and deduplicating equivalent matches**
4. **RDKit substructure matching: chemistry-aware behavior**
5. **MCS with RDKit: maximum common substructure as alignment**
6. **From MCS to atom maps: building a practical correspondence**
7. **RDKit vs NetworkX morphisms: comparison and pitfalls**
8. **Discussion, quiz, and references**


In [ ]:
import rdkit
from rdkit import Chem
from rdkit.Chem import rdFMCS
import networkx as nx
import pandas as pd
from pathlib import Path

print("RDKit version:", rdkit.__version__)
print("NetworkX version:", nx.__version__)


RDKit version: 2025.09.3
NetworkX version: 3.6.1


In [3]:
from typing import Dict
import networkx as nx
from rdkit import Chem
import rdkit

In [4]:
# RDKit -> NetworkX

def mol_to_graph(mol: Chem.Mol, include_implicit_h: bool = True) -> nx.Graph:
    """
    Convert RDKit Mol -> typed NetworkX graph.

    :param mol: RDKit Mol (assumed sanitized).
    :param include_implicit_h: If True, store total H count per atom as ``total_h``.
    :returns: networkx.Graph with atom/bond labels as node/edge attributes.
    """
    G = nx.Graph()

    for atom in mol.GetAtoms():
        i = atom.GetIdx()
        attrs: Dict[str, object] = {
            "symbol": atom.GetSymbol(),
            "formal_charge": int(atom.GetFormalCharge()),
            "aromatic": bool(atom.GetIsAromatic()),
            "chiral_tag": str(atom.GetChiralTag()),
        }
        if include_implicit_h:
            attrs["total_h"] = int(atom.GetTotalNumHs())
        G.add_node(i, **attrs)

    for bond in mol.GetBonds():
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()
        order = int(round(bond.GetBondTypeAsDouble()))
        G.add_edge(
            u, v,
            order=order,
            aromatic=bool(bond.GetIsAromatic()),
            in_ring=bool(bond.IsInRing()),
        )

    G.graph["source"] = "rdkit"
    G.graph["rdkit_version"] = rdkit.__version__
    return G


In [5]:
# NetworkX to rdkit
def graph_to_mol(G: nx.Graph, make_explicit_h: bool = False) -> Chem.Mol:
    """
    Reconstruct RDKit Mol from typed NetworkX graph (inverse of ``mol_to_graph`` up to sanitization).

    :param G: Typed molecular graph produced by ``mol_to_graph``.
    :param make_explicit_h: If True and ``total_h`` exists, add explicit H atoms (best-effort).
    :returns: Sanitized RDKit Mol.
    """
    rw = Chem.RWMol()
    nx_to_rdk: Dict[int, int] = {}

    # atoms
    for node in sorted(G.nodes()):
        n = G.nodes[node]
        atom = Chem.Atom(n.get("symbol", "C"))
        atom.SetFormalCharge(int(n.get("formal_charge", 0)))
        if n.get("aromatic", False):
            atom.SetIsAromatic(True)

        ch_tag = n.get("chiral_tag")
        if ch_tag and ch_tag != "CHI_UNSPECIFIED":
            try:
                atom.SetChiralTag(getattr(Chem.rdchem.ChiralType, ch_tag))
            except Exception:
                pass  # best-effort only

        nx_to_rdk[node] = rw.AddAtom(atom)

    # bonds
    for u, v, e in G.edges(data=True):
        order = int(e.get("order", 1))
        btype = {
            1: Chem.rdchem.BondType.SINGLE,
            2: Chem.rdchem.BondType.DOUBLE,
            3: Chem.rdchem.BondType.TRIPLE,
        }.get(order, Chem.rdchem.BondType.SINGLE)
        rw.AddBond(nx_to_rdk[u], nx_to_rdk[v], btype)

    mol = rw.GetMol()

    # optional explicit H
    if make_explicit_h:
        for node, rdk_idx in nx_to_rdk.items():
            total_h = G.nodes[node].get("total_h")
            if total_h is None:
                continue
            atom = mol.GetAtomWithIdx(rdk_idx)
            current_h = sum(1 for n in atom.GetNeighbors() if n.GetSymbol() == "H")
            for _ in range(max(int(total_h) - current_h, 0)):
                h_idx = mol.AddAtom(Chem.Atom("H"))
                mol.AddBond(rdk_idx, h_idx, Chem.rdchem.BondType.SINGLE)

    Chem.SanitizeMol(mol)
    return mol


In [ ]:
from networkx.algorithms import isomorphism as iso

def node_match(n1, n2):
    # Minimal chemical identity (extend in later notebooks if needed)
    return (
        n1.get("symbol") == n2.get("symbol")
        and int(n1.get("formal_charge", 0)) == int(n2.get("formal_charge", 0))
        and bool(n1.get("aromatic", False)) == bool(n2.get("aromatic", False))
    )

def edge_match(e1, e2):
    return (
        int(e1.get("order", 1)) == int(e2.get("order", 1))
        and bool(e1.get("aromatic", False)) == bool(e2.get("aromatic", False))
    )

def enumerate_automorphisms(G: nx.Graph):
    GM_self = iso.GraphMatcher(G, G, node_match=node_match, edge_match=edge_match)
    return list(GM_self.isomorphisms_iter())

def compute_orbits_from_automorphisms(G: nx.Graph, automorphisms=None):
    # Copied from S01 but kept small here for self-containment.
    if automorphisms is None:
        automorphisms = enumerate_automorphisms(G)

    parent = {v: v for v in G.nodes()}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for auto in automorphisms:
        for v, fv in auto.items():
            union(v, fv)

    orbits = {}
    for v in G.nodes():
        r = find(v)
        orbits.setdefault(r, set()).add(v)

    return sorted(orbits.values(), key=lambda s: min(s))


## 1. Subgraph isomorphism (pattern → host)

In rule-based reaction modeling, we repeatedly solve the **pattern-in-host** query:

$$
\text{Does a pattern graph } P \text{ occur inside a host graph } G?
\quad \text{If yes, what are the embeddings?}
$$

Formally, a **subgraph isomorphism** is an **injective, label-preserving graph morphism**

$$
f : V(P) \hookrightarrow V(G)
$$

such that:

- **Atom (node) labels are preserved**
  $$a_G(f(v)) = a_P(v)\quad \forall v \in V(P)$$

- **Bond existence and bond types are preserved**
  $$uv \in E(P)\ \Rightarrow\ f(u)f(v) \in E(G), \quad b_G(f(u)f(v)) = b_P(uv)$$

Intuitively, \(f\) is a **typed monomorphism**: it embeds the pattern into the host
without collisions (injective), while respecting chemical identity (types).

### Practical note: many matches due to symmetry
Even when the chemical occurrence is “the same”, symmetric graphs can admit many
equally valid embeddings:

- **Host symmetry** (automorphisms of \(G\)) produces multiple placements.
- **Pattern symmetry** (automorphisms of \(P\)) produces multiple equivalent mappings.
- If both are symmetric, matches can multiply combinatorially.

NetworkX exposes this via:

- `GraphMatcher.subgraph_isomorphisms_iter()` — enumerates all injective embeddings
  that satisfy `node_match` and `edge_match`.

For downstream tasks (reaction center extraction, rule application, deduplication),
we often need to **post-process** these matches to remove symmetry-equivalent
embeddings, typically by orbit-based canonicalization or choosing a canonical
representative embedding.

In [58]:
from __future__ import annotations

from collections import Counter
from typing import Dict, List, Iterable, Tuple
import networkx as nx
from networkx.algorithms import isomorphism as iso
from rdkit import Chem


def nx_subgraph_matches(
    host_G: nx.Graph,
    pattern_G: nx.Graph,
    *,
    invert: bool = True,
) -> List[Dict]:
    """
    Enumerate subgraph isomorphisms of `pattern_G` inside `host_G`.

    Notes
    -----
    NetworkX's GraphMatcher(host, pattern).subgraph_isomorphisms_iter()
    yields mappings of the form:  host_node -> pattern_node  (G1 -> G2).

    If you want the more intuitive direction (pattern -> host), set `invert=True`.
    """
    GM = iso.GraphMatcher(
        host_G,
        pattern_G,
        node_match=node_match,
        edge_match=edge_match,
    )

    out: List[Dict] = []
    for m_host_to_pat in GM.subgraph_isomorphisms_iter():
        if invert:
            out.append({p: h for h, p in m_host_to_pat.items()})
        else:
            out.append(m_host_to_pat)
    return out


# --- Symmetry-heavy example: benzene pattern in naphthalene host ---
# Pattern is highly symmetric (|Aut|=12), and the host contains two benzene rings.
# => many raw embeddings (symmetry variants)

pattern_mol = Chem.MolFromSmiles("c1ccccc1")          # benzene (symmetric)
host_mol    = Chem.MolFromSmiles("c1ccc2ccccc2c1")    # naphthalene (symmetric)

pattern_G = mol_to_graph(pattern_mol)
host_G    = mol_to_graph(host_mol)

matches = nx_subgraph_matches(host_G, pattern_G, invert=True)  # pattern_idx -> host_idx

print("Raw subgraph isomorphisms (pattern -> host):", len(matches))


Raw subgraph isomorphisms (pattern -> host): 24


### Deduplication of subgraph embeddings

Raw subgraph matches often contain many symmetry-equivalent embeddings.  
We present a minimal, renderer-friendly description of the **host-image** deduplication strategy.

Let \(m : V(P)\to V(G)\) be a pattern→host mapping. Define the **image** of \(m\) as the set of host nodes touched by the embedding:

$$
\mathrm{img}(m)\;=\;\{\,m(v)\;|\;v\in V(P)\,\}\subseteq V(G).
$$

We deduplicate embeddings by grouping all mappings that share the same image.  
Operationally we use the sorted tuple of `img(m)` as a canonical key:

$$
\text{key}(m) \;=\; \text{tuple}(\mathrm{sorted}(\mathrm{img}(m))).
$$

This collapses symmetry variants that differ only by a permutation of pattern nodes (i.e., different bijections from the same image) and yields the distinct *placements* of the pattern inside the host.

In [69]:
from collections import defaultdict
from typing import Dict, Iterable, List, Mapping, Tuple, Union, Optional


def _build_node_to_rep(
    orbits: Union[Iterable[Iterable[int]], Mapping[int, int], None]
) -> Dict[int, int]:
    """Convert `orbits` into a mapping node -> representative."""
    if orbits is None:
        return {}
    if isinstance(orbits, Mapping):
        return dict(orbits)
    node_to_rep: Dict[int, int] = {}
    for orbit in orbits:
        orbit = set(orbit)
        if not orbit:
            continue
        rep = min(orbit)
        for v in orbit:
            node_to_rep[v] = rep
    return node_to_rep


def dedup_by_host_image_with_orbits(
    matches: Iterable[Dict[int, int]],
    *,
    orbits: Union[Iterable[Iterable[int]], Mapping[int, int], None] = None,
    pattern_node_order: Optional[Iterable[int]] = None,
    return_groups: bool = False,
) -> Union[List[Dict[int, int]], Dict[Tuple[int, ...], List[Dict[int, int]]]]:
    """
    Deduplicate pattern->host mappings by canonical host-image (with optional orbit reps).

    Parameters
    ----------
    matches : iterable of dict
        Each mapping is pattern_node -> host_node.
    orbits : iterable of iterables OR mapping node->rep OR None
        If iterable-of-iterables, representative for an orbit is min(orbit).
        If mapping, it should be node -> representative.
        If None, no orbit canonicalization is applied.
    pattern_node_order : iterable of pattern node ids, optional
        Order used to form the per-mapping tuple for selecting the representative.
        If None, inferred deterministically from the first mapping (sorted keys).
    return_groups : bool (default False)
        If False (default) return a list of representative mappings (one per canonical class).
        If True return the full dict: canonical_key -> list[mappings].

    Returns
    -------
    List[dict] or Dict[tuple, list]
        See `return_groups` description.
    """
    matches = list(matches)  # materialize to allow multiple passes
    if not matches:
        return {} if return_groups else []

    node_to_rep = _build_node_to_rep(orbits)

    # infer or validate pattern node order
    if pattern_node_order is None:
        # Use sorted keys of the first mapping (deterministic)
        pattern_node_order = tuple(sorted(matches[0].keys()))
    else:
        pattern_node_order = tuple(pattern_node_order)

    # bucket by canonical key (sorted canonicalized host nodes)
    buckets: Dict[Tuple[int, ...], List[Dict[int, int]]] = defaultdict(list)
    for m in matches:
        canonical_nodes = (node_to_rep.get(h, h) for h in m.values())
        key = tuple(sorted(canonical_nodes))
        buckets[key].append(m)

    # if user requested full groups, return them (deterministic ordering by key)
    if return_groups:
        return {k: buckets[k] for k in sorted(buckets.keys())}

    # otherwise pick a deterministic representative per bucket:
    # representative = mapping with smallest tuple (m[p] for p in pattern_node_order)
    representatives: List[Dict[int, int]] = []
    for key in sorted(buckets.keys()):
        group = buckets[key]
        # compute lexicographic tuple for each mapping according to pattern_node_order
        def ordering_tuple(m: Dict[int, int]) -> Tuple[int, ...]:
            return tuple(m[p] for p in pattern_node_order)
        rep = min(group, key=ordering_tuple)
        representatives.append(rep)

    return representatives


In [67]:
# 1. compute host orbits (optional — useful if you later want to collapse host symmetry)
host_autos = enumerate_automorphisms(host_G)                       # φ : host_node -> host_node
host_orbits = compute_orbits_from_automorphisms(host_G, host_autos)  # list of sets
print("Host orbits:", host_orbits)
print()

# 2. dedup by host-image (default: returns representatives list)
representatives = dedup_by_host_image_with_orbits(matches)  # list of rep mappings
print("Number of representatives:", len(representatives))
for i, rep in enumerate(representatives, start=1):
    print(f" representative #{i}:", rep)
print()

# 3. (optional) show full groups and multiplicities to verify symmetry inflation
groups = dedup_by_host_image_with_orbits(matches, return_groups=True)  # dict: key -> list[mappings]
print("Distinct placements by host-image (groups):", len(groups))
for key, group in groups.items():
    print(" placement key:", key, " multiplicity:", len(group))
    print("  example mapping:", group[0])
print()

# 4. (optional) collapse placements modulo host automorphisms (use host_orbits)
groups_mod_host = dedup_by_host_image_with_orbits(matches, orbits=host_orbits, return_groups=True)
print("Unique classes modulo host automorphisms:", len(groups_mod_host))
for key, group in groups_mod_host.items():
    print(" canonical_key (orbits reps):", key, " multiplicity:", len(group))
    print("  example mapping:", group[0])


Host orbits: [{0, 1, 5, 6}, {9, 2, 4, 7}, {8, 3}]

Number of representatives: 2
 representative #1: {0: 0, 1: 1, 2: 2, 3: 3, 4: 8, 5: 9}
 representative #2: {0: 3, 1: 4, 2: 5, 3: 6, 4: 7, 5: 8}

Distinct placements by host-image (groups): 2
 placement key: (0, 1, 2, 3, 8, 9)  multiplicity: 12
  example mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 8, 5: 9}
 placement key: (3, 4, 5, 6, 7, 8)  multiplicity: 12
  example mapping: {0: 3, 1: 8, 2: 7, 3: 6, 4: 5, 5: 4}

Unique classes modulo host automorphisms: 1
 canonical_key (orbits reps): (0, 0, 2, 2, 3, 3)  multiplicity: 24
  example mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 8, 5: 9}


### Exercise: Deduplicate modulo pattern automorphisms

### Q5 — Implement `dedup_by_pattern_image_with_orbits`

**Goal.**  
Implement a function `dedup_by_pattern_image_with_orbits(matches, pattern_autos, ...)` that groups/filters pattern→host mappings **modulo pattern automorphisms**.

Intuitively, two mappings \(m_1,m_2: P \to H\) are equivalent if there exists a pattern automorphism \(\varphi \in \mathrm{Aut}(P)\) such that
$$
m_2 \;=\; m_1 \circ \varphi .
$$
Equivalently, \(m_1\) and \(m_2\) differ only by a permutation of the pattern nodes.

<details> <summary><b>Solution:</b></summary>

```python
from collections import defaultdict
from typing import Dict, Iterable, List, Tuple, Optional, Union

def dedup_by_pattern_image_with_orbits(
    matches: Iterable[Dict[int,int]],
    *,
    pattern_autos: Iterable[Dict[int,int]],
    pattern_node_order: Optional[Iterable[int]] = None,
    return_groups: bool = False,
) -> Union[List[Dict[int,int]], Dict[Tuple[int,...], List[Dict[int,int]]]]:
    matches = list(matches)
    if not matches:
        return {} if return_groups else []

    # deterministic pattern node order
    if pattern_node_order is None:
        pattern_node_order = tuple(sorted(matches[0].keys()))
    else:
        pattern_node_order = tuple(pattern_node_order)

    # pre-list autos for repeated use
    autos = list(pattern_autos)

    # helper: canonical tuple for mapping m under all pattern automorphisms
    def canonical_tuple_for_mapping(m: Dict[int,int]) -> Tuple[int, ...]:
        best = None
        for phi in autos:
            tup = tuple(m[phi[p]] for p in pattern_node_order)
            if best is None or tup < best:
                best = tup
        return best

    # bucket by canonical tuple
    buckets: Dict[Tuple[int,...], List[Dict[int,int]]] = defaultdict(list)
    for m in matches:
        key = canonical_tuple_for_mapping(m)
        buckets[key].append(m)

    # deterministic ordering of keys
    ordered_keys = sorted(buckets.keys())

    if return_groups:
        return {k: buckets[k] for k in ordered_keys}

    # select deterministic representative per bucket:
    # pick mapping with smallest tuple according to pattern_node_order
    def ordering_tuple(m: Dict[int,int]) -> Tuple[int, ...]:
        return tuple(m[p] for p in pattern_node_order)

    representatives: List[Dict[int,int]] = []
    for k in ordered_keys:
        group = buckets[k]
        rep = min(group, key=ordering_tuple)
        representatives.append(rep)

    return representatives

```

## 2. MCS (Maximum Common Substructure) — RDKit + NetworkX views

The **Maximum Common Substructure (MCS)** problem asks for the largest subgraph that two molecular graphs share.
It is a core *alignment primitive* used in similarity, substructure transfer, and as a common starting point for **atom mapping**.

---

### 6.1 Formal definition (typed molecular graphs)

Let
$$
G = (V_G, E_G, a_G, b_G)
\quad \text{and} \quad
H = (V_H, E_H, a_H, b_H)
$$
be **typed molecular graphs**, where
\( a(\cdot) \) assigns atom labels (element, charge, aromaticity) and
\( b(\cdot) \) assigns bond labels (bond order, aromaticity).

A graph \( S \) is a **common subgraph** of \( G \) and \( H \) if there exist
**injective typed morphisms** (subgraph embeddings)


$$
f: S \hookrightarrow G, \qquad g: S \hookrightarrow H
$$

such that labels and bonds are preserved (same predicates as subgraph isomorphism).

An **MCS** is any common subgraph \(S^\*\) that maximizes a size objective:

$$
S^* \in \arg\max_{S}
\left(
|V(S)| \;\text{or}\; w_V |V(S)| + w_E |E(S)|
\right)
\quad
\text{subject to } S \hookrightarrow G \text{ and } S \hookrightarrow H.
$$


**Notes.**
- The MCS need not be unique (multiple maximum solutions may exist).
- Constraints (ring-only matching, atom types, bond types, chirality) change the feasible set and thus the MCS.

---

### 6.2 RDKit view: MCS as a SMARTS pattern + match lists

RDKit provides a practical MCS solver:

- `rdFMCS.FindMCS([mol1, mol2], ...)` returns an object whose `smartsString`
  encodes a **query substructure** \(Q\) (SMARTS) intended to represent a largest shared substructure under constraints.

We then compute embeddings of the SMARTS query into each molecule:

$$
\mathrm{Match}_G(Q) = \{\, f_i : V(Q)\hookrightarrow V(G)\,\}, \qquad
\mathrm{Match}_H(Q) = \{\, g_j : V(Q)\hookrightarrow V(H)\,\}.
$$

In practice:
- `mol.GetSubstructMatches(query)` returns many embeddings (symmetry variants).
- When multiple matchings exist, choosing a canonical representative (or a best-scoring one) is a separate step.

**Interpretation.**  
RDKit’s MCS output gives you:
1) a *candidate* maximum common substructure (as a query), and  
2) potentially many embeddings in each molecule.

---

### 6.3 NetworkX view: MCS as a maximum common *subgraph isomorphism*

When we convert molecules to NetworkX graphs, MCS corresponds to finding a largest typed graph \(S\)
that is simultaneously subgraph-isomorphic to both graphs.

Conceptually:

$$
S \subseteq G,\; S \subseteq H
\quad\Longleftrightarrow\quad
\exists\, f: S\hookrightarrow G,\; \exists\, g: S\hookrightarrow H.
$$

In the NX world, this is typically attacked by:
- searching over candidate node/bond subsets, or
- using MCS heuristics (e.g., expand from seeds; branch-and-bound; constraint propagation),
because exact MCS is NP-hard.

**Why still use NX here?**
- You can enforce *your* exact chemical typing predicates (`node_match`, `edge_match`).
- You can integrate symmetry handling (automorphism orbits) and custom constraints.
- You can expose intermediate states for teaching (what gets pruned, what expands).

---

### 6.4 Symmetry and non-uniqueness (important for mapping)

Both RDKit and NetworkX share the same caveats:

- **Multiple maximum solutions:** the same maximum subgraph size can be achieved by different subgraphs.
- **Many embeddings:** even one fixed maximum subgraph can have many matches due to molecular symmetry.
- **Implication:** MCS is a useful starting point for atom mapping, but not sufficient on its own — a tie-breaking or scoring rule is still required.

---

### 6.5 Practical pipeline (RDKit ↔ NX)

A common didactic workflow is:

1. **RDKit MCS**
   - compute a SMARTS query \(Q\) using `rdFMCS.FindMCS`.
2. **RDKit embeddings**
   - enumerate `GetSubstructMatches(Q)` for each molecule.
3. **NX analysis**
   - convert selected embeddings to NX node correspondences,
   - optionally deduplicate symmetry-equivalent matches using orbits,
   - use the resulting correspondence as a seed for atom mapping or reaction-center extraction.

This makes MCS an excellent bridge between *chemistry-native toolkits* (RDKit) and *graph-theoretic control* (NetworkX).

In [9]:
from rdkit import Chem
from rdkit.Chem import rdFMCS


def rdkit_mcs_smarts(
    mols,
    timeout=10,
    ringMatchesRingOnly=True,
    atomCompare=rdFMCS.AtomCompare.CompareElements,
    bondCompare=rdFMCS.BondCompare.CompareOrder,
):
    res = rdFMCS.FindMCS(
        mols,
        atomCompare=atomCompare,
        bondCompare=bondCompare,
        ringMatchesRingOnly=ringMatchesRingOnly,
        timeout=timeout,
    )
    return res.smartsString


# Example molecules
m1 = Chem.MolFromSmiles("Oc1ccccc1")   # phenol
m2 = Chem.MolFromSmiles("Cc1ccccc1")   # toluene

mcs_smarts = rdkit_mcs_smarts([m1, m2])
print("MCS SMARTS:", mcs_smarts)

mcs_mol = Chem.MolFromSmarts(mcs_smarts)
print("MCS matches in m1:", m1.GetSubstructMatches(mcs_mol))
print("MCS matches in m2:", m2.GetSubstructMatches(mcs_mol))


MCS SMARTS: [#6]1:&@[#6]:&@[#6]:&@[#6]:&@[#6]:&@[#6]:&@1
MCS matches in m1: ((1, 2, 3, 4, 5, 6),)
MCS matches in m2: ((1, 2, 3, 4, 5, 6),)


## 3. RDKit vs NetworkX morphisms: comparison

- **RDKit substructure matching** is chemistry-aware and supports internal symmetry handling via `uniquify=True`.
- **NetworkX subgraph isomorphism** is pure typed-graph matching and often returns all symmetric variants unless you dedupe.

We compare match counts and host-atom sets for the same (host, pattern) pair.

In [ ]:
def rdkit_substruct_matches(host: Chem.Mol, pattern: Chem.Mol, uniquify: bool = True):
    return host.GetSubstructMatches(pattern, uniquify=uniquify)

def compare_rdkit_vs_nx(host_smiles: str, pattern_smiles: str):
    host_m = Chem.MolFromSmiles(host_smiles)
    pattern_m = Chem.MolFromSmiles(pattern_smiles)

    rd_matches = rdkit_substruct_matches(host_m, pattern_m, uniquify=True)

    host_G = mol_to_graph(host_m)
    pattern_G = mol_to_graph(pattern_m)
    nx_matches = nx_subgraph_matches(host_G, pattern_G)

    rd_hostsets = {tuple(sorted(t)) for t in rd_matches}
    nx_hostsets = {tuple(sorted(m.values())) for m in nx_matches}

    print("Host:", host_smiles)
    print("Pattern:", pattern_smiles)
    print("RDKit matches (uniquify=True):", len(rd_matches))
    print("NetworkX matches (raw):", len(nx_matches))
    print("NetworkX unique hostsets:", len(nx_hostsets))
    print("RDKit hostsets:", rd_hostsets)
    print("NX hostsets:", nx_hostsets)
    print("NX extra (not in RDKit hostsets):", nx_hostsets - rd_hostsets)
    print("RDKit-only (if any):", rd_hostsets - nx_hostsets)

compare_rdkit_vs_nx("Oc1ccccc1", "c1ccccc1")


## 4. Discussion (what to remember)

- A **typed graph morphism** formalizes structure- and attribute-preserving maps between graphs.
- Our **typed molecular graphs** use a minimal attribute schema (`symbol`, `formal_charge`, `aromatic`, `order`) to define what “same” means.
- **Round-trip conversion** (RDKit → NetworkX → RDKit) is valuable for debugging and peer review; we preserve heavy-atom topology, but exact RDKit internal state may differ.
- **Automorphisms** describe symmetries; they inflate match enumeration. Deduplicate (e.g. by host-atom set) to prevent combinatorial explosion.
- **Subgraph isomorphism** is the core operation for rule application later in SynEdu.
- **MCS** is a chemistry-aware alignment primitive, but it is heuristic and sometimes non-unique; always log settings and timeouts.
- **RDKit vs NetworkX**:
  - RDKit: chemistry-aware SMARTS matching, built-in `uniquify`.
  - NetworkX: full control over attributes and morphism semantics; you manage deduplication and interpretation.

## 5. Quiz · Graph Matching Fundamentals

Answer the following questions using **both chemical intuition and formal graph language**.

---

### 1. Typed graph morphism

In one or two sentences, define a **typed graph morphism**.

$$
f : G \rightarrow H
$$

- What objects does f map?
- Which **atom** and **bond** properties must be preserved?
- Give one example of a mapping that would be **invalid** in chemistry.


---

### 2. Isomorphism  
What **additional requirement** must a graph morphism satisfy to become an  
**isomorphism**?

- How does this relate to the idea of *two molecules having the same structure*?

---

### 3. Automorphism and symmetry  
What is an **automorphism** of a molecular graph?

- Why do symmetric molecules (e.g. benzene) have **many automorphisms**?
- Why do automorphisms cause **duplicate subgraph matches** during matching?

---

### 4. Deduplicating subgraph matches  
Subgraph matching often returns many equivalent matches.

- Explain how using **sets of host atom indices** can be used to
  **deduplicate** equivalent matches.
- Why does this work even when atom ordering differs?

---

### 5. RDKit vs NetworkX (practice vs theory)  
Give **one practical advantage** of each approach:

- **RDKit** substructure matching  
- **NetworkX** graph matching  

In which situations would you prefer one over the othe

## 6. References and further reading

- RDKit documentation: https://www.rdkit.org/docs/  
- RDKit Book: https://www.rdkit.org/docs/Book.html  
- NetworkX documentation: https://networkx.org/documentation/stable/  
- NetworkX isomorphism: https://networkx.org/documentation/stable/reference/algorithms/isomorphism.html  
- RDKit MCS (rdFMCS): https://www.rdkit.org/docs/source/rdkit.Chem.rdFMCS.html  